# **OCR Live Demo**

End-to-end text detection, recognition, and speech synthesis for visually impaired users.

**Stack:**
- Detection   : db_resnet50 (docTR)
- Recognition : TrOCR base-printed (fine-tuned on COCO-Text)
- Confidence gate : 0.75
- TTS         : gTTS (cloud)

**Final metrics (Phase 6b, 1000 test images):**
- CER : 0.1998  (pretrained baseline: 0.3337)
- P95 latency : within 2s budget on 99.7% of images
- Gate pass rate : 42.1%

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')


: 

: 

In [ ]:
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

pip('python-doctr[torch]')
pip('transformers')
pip('pyyaml')
pip('gtts')
pip('typer==0.9.0')
pip('click==8.1.7')

print('Done — restart runtime, then continue from cell 3.')

In [2]:
!pip install gradio python-doctr[torch] transformers gtts pyyaml editdistance typer==0.9.0 click==8.1.7 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 8.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 37.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 75.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.6/313.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 82.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/

In [5]:
import sys, io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from IPython.display import Audio, display
import torch

DRIVE_ROOT   = Path('/content/drive/MyDrive/vision-ocr-accessibility-assistant')
PROJECT_ROOT = DRIVE_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_CACHE  = DRIVE_ROOT / 'models/trocr'
FINETUNE_DIR = DRIVE_ROOT / 'models/trocr-finetuned'

from src.pipeline import Pipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

pipe = Pipeline(
    device=DEVICE,
    tts_backend='cloud',
    model_cache_dir=str(MODEL_CACHE),
    recognition_checkpoint=str(FINETUNE_DIR)
)
print('Pipeline ready ')

In [3]:
!pip install gradio huggingface_hub --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 12.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 12.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-adk 1.27.1 requires websockets<16.0.0,>=15.0.1, but you have websockets 11.0.3 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


In [4]:
!python /content/drive/MyDrive/vision-ocr-accessibility-assistant/app.py

Project root : /content/drive/MyDrive/vision-ocr-accessibility-assistant
Fine-tuned   : /content/drive/MyDrive/vision-ocr-accessibility-assistant/models/trocr-finetuned
Loading pipeline on cuda...
Loading detector (db_resnet50) on cuda...
102022144it [00:00, 230127460.22it/s]                  
Detector ready
Loading recognizer: /content/drive/MyDrive/vision-ocr-accessibility-assistant/models/trocr-finetuned on cuda...
Loading weights: 100% 480/480 [00:38<00:00, 12.33it/s, Materializing param=encoder.pooler.dense.weight]                                   
Recognizer ready ✓  (334M parameters)

Pipeline ready ✓
  Confidence gate : 0.75
  Latency budget  : 2.0s
  TTS backend     : cloud
Pipeline ready ✓
/content/drive/MyDrive/vision-ocr-accessibility-assistant/app.py:145: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="OCR Accessibility Pipeline", 

## 2. Visualization Utility

In [ ]:
def visualize_result(img_path, result):
    """
    Display image with bounding box overlay.
    Green = spoken (confidence >= gate)
    Red   = silenced (confidence < gate)
    """
    img = Image.open(img_path).convert('RGB')
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)

    for r in result['gated_results']:
        x, y, w, h = r['box']
        color  = '#27ae60' if not r['gated'] else '#e74c3c'
        label  = f"{r['text']} ({r['confidence']:.2f})"
        rect   = patches.Rectangle(
            (x, y), w, h,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x, y - 4, label, fontsize=8, color=color,
                fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))

    spoken   = [r['text'] for r in result['gated_results'] if not r['gated']]
    silenced = [r['text'] for r in result['gated_results'] if r['gated']]

    ax.set_title(
        f"Spoken: {spoken}\n"
        f"Silenced: {silenced}\n"
        f"Latency: {result['latency']['detection_s']*1000:.1f}ms detection + "
        f"{result['latency']['recognition_s']*1000:.1f}ms recognition = "
        f"{(result['latency']['detection_s']+result['latency']['recognition_s'])*1000:.1f}ms total",
        fontsize=10, loc='left'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.show()

print('Visualization ready ✓')

## 3. Single Image Demo — Upload Your Own Image

In [ ]:
from google.colab import files

uploaded = files.upload()
img_path = list(uploaded.keys())[0]

print(f'Running pipeline on: {img_path}')
result = pipe.run(img_path)

spoken = [r['text'] for r in result['gated_results'] if not r['gated']]
print(f'\nSpoken regions : {spoken}')
print(f'Total regions  : {result["n_boxes"]}')
print(f'Spoken         : {result["n_spoken"]}')
print(f'Silenced       : {result["n_silenced"]}')

visualize_result(img_path, result)

if result['tts_output']:
    print('\nPlaying TTS output:')
    display(Audio(result['tts_output'], autoplay=True))
else:
    print('\nNo text passed the confidence gate — nothing to speak.')

## 4. Batch Demo — Sample Test Images

In [ ]:
import json, random
from collections import defaultdict

RESULTS_DIR = DRIVE_ROOT / 'outputs/results'
IMG_DIR     = DRIVE_ROOT / 'data/raw/benchmark_images'
N_DEMO      = 6

with open(RESULTS_DIR / '04_test_sample.json') as f:
    test_data = json.load(f)

# Sample 2 images from each legibility bin for variety
bins = defaultdict(list)
for s in test_data:
    bins[s.get('legibility_bin', 'low')].append(s)

random.seed(12)
demo_samples = []
for b in ['high', 'medium', 'low']:
    available = [s for s in bins[b] if (IMG_DIR / s['file_name']).exists()]
    n = N_DEMO // 3
    demo_samples.extend(random.sample(available, min(n, len(available))))

# Run pipeline on all demo images
demo_results = []
print(f'Running pipeline on {len(demo_samples)} images...')
for i, s in enumerate(demo_samples):
    img_path = IMG_DIR / s['file_name']
    if not img_path.exists(): continue
    result = pipe.run(img_path)
    demo_results.append((img_path, s, result))
    print(f'  {i+1}/{len(demo_samples)} done')

# Display all results in a grid
n_rows = len(demo_results)
fig, axes = plt.subplots(n_rows, 2, figsize=(16, 6 * n_rows))
axes = np.array(axes).reshape(n_rows, 2)  # ensure always 2D

for row, (img_path, s, result) in enumerate(demo_results):
    img = Image.open(img_path).convert('RGB')

    # Left: original image
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{s['file_name']} [{s.get('legibility_bin', '')}]", fontsize=9)
    axes[row, 0].axis('off')

    # Right: annotated image
    axes[row, 1].imshow(img)
    for r in result['gated_results']:
        x, y, w, h = r['box']
        color = '#27ae60' if not r['gated'] else '#e74c3c'
        rect  = patches.Rectangle((x, y), w, h,
                                    linewidth=2, edgecolor=color, facecolor='none')
        axes[row, 1].add_patch(rect)
        axes[row, 1].text(x, y - 4, f"{r['text']} ({r['confidence']:.2f})",
                           fontsize=7, color=color, fontweight='bold',
                           bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))

    spoken = [r['text'] for r in result['gated_results'] if not r['gated']]
    lat_ms = (result['latency']['detection_s'] + result['latency']['recognition_s']) * 1000
    axes[row, 1].set_title(f"Spoken: {spoken} | {lat_ms:.0f}ms", fontsize=9)
    axes[row, 1].axis('off')

plt.suptitle('OCR Accessibility Pipeline — Batch Demo', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# TTS audio players
print('\nTTS outputs:')
for img_path, s, result in demo_results:
    if result['tts_output']:
        print(f'  {s["file_name"]}')
        display(Audio(result['tts_output'], autoplay=False))

# Phase 7 — Demo Observations

## Limitations Identified

**1. Vertical text not readable**
The pipeline fails on vertical text. DBNet detects the bounding box correctly
but TrOCR produces incorrect output since it was pretrained on horizontal
printed text only. The recognized text is either garbage or low confidence
and gets silenced by the gate.

**2. Large spaced capital letters split into individual words**
On large signage where letters are widely spaced, DBNet draws one box per
letter instead of one box per word. The pipeline then speaks each letter
individually rather than the full word.

**3. Texture patterns falsely detected as text**
High-contrast repeating textures such as a patterned mat are detected as
text regions by DBNet. TrOCR typically produces low-confidence output on
these which the confidence gate catches, but not always.

## Notes
These are known limitations of the pretrained DBNet and TrOCR stack on
real-world scene text. They are identified as future work and do not affect
performance on the primary target use case of printed scene text on signs,
labels, and storefronts.